# Controle ótimo de nível de tanques — COS360

Este notebook reúne os dois scripts MATLAB enviados: controle de **um tanque** e controle de **dois tanques**. Os códigos foram preservados como células MATLAB, incluindo as funções auxiliares que aparecem no final de cada script.

---
# 1. Controle de um tanque

Arquivo de origem: `OTIMIZACAO_1tanque.m`.

O script configura a comunicação Modbus, os parâmetros da planta, o perfil de demanda, executa o controle SQP em horizonte móvel e apresenta métricas e gráficos do experimento. O Flow Meter e o potenciômetro de setpoint aparecem no mapeamento de registradores do próprio arquivo.

In [ ]:
%% controle_1_tanque.m
% Projeto COS360 - Otimizacao (PESC/COPPE/UFRJ)
% Controle Otimo Nao Linear (PQS / SQP) para 1 Tanque no Factory I/O via Modbus TCP
% Servidor Modbus TCP/IP Factory I/O: 192.168.15.117:502 | Slave ID: 1
%
% Entrada de Demanda em LITROS (L/s ou L/min):
%   - Configure o vetor 'Q_dem_litros' na Secao 3 diretamente em Litros/s (ou Litros/min).
%   - Faixa fisica recomendada para 1 Tanque:
%       * Vazao minima (valvula em 2 V):  ~46.3 L/s  (2775 L/min)
%       * Vazao maxima entrada (10 V):   ~230.9 L/s (13851 L/min)
%       * Vazao maxima saida (h = 2 m):  ~268.7 L/s (limite fisico: 354.3 L/s)

clear; clc; close all;

%% 1. Configuracao de Comunicacao Modbus TCP (Factory I/O)
IP_PLANTA            = '192.168.15.117';
PORTA_MODBUS         = 502;
SLAVE_ID             = 1;
USAR_MODBUS          = true;   % true: conecta ao Factory I/O | false: simulacao local RK4

% Fator de escala configurado no driver Modbus TCP/IP Server do Factory I/O (padrao = 100)
ESCALA_FIO           = 100.0;
ATUALIZAR_PAINEL_FIO = true;   % Atualiza displays SP e PV e luzes Start/Stop no painel

% Enderecos Modbus no MATLAB (base 1 = endereco Factory I/O + 1)
IREG_LEVEL_METER     = 1;      % Input Reg 0: Level meter (Tanque central)
IREG_FLOW_METER      = 2;      % Input Reg 1: Flow meter  (Tanque central)
IREG_SETPOINT_POT    = 3;      % Input Reg 2: Setpoint (potenciometro painel, opcional)
INP_FIO_RUNNING      = 4;      % Input 3:     FACTORY I/O (Running)

HREG_FILL_VALVE      = 1;      % Holding Reg 0: Fill valve (ue)
HREG_DISC_VALVE      = 2;      % Holding Reg 1: Discharge valve (us)
HREG_PANEL_SP        = 3;      % Holding Reg 2: SP (display painel)
HREG_PANEL_PV        = 4;      % Holding Reg 3: PV (display painel)

COIL_START_LIGHT     = 1;      % Coil 0: Start light
COIL_STOP_LIGHT      = 3;      % Coil 2: Stop light

%% 2. Parametros Fisicos e Restricoes da Planta (Modelo Matematico)
params.A          = pi;          % Area transversal [m^2] (diametro = 2.0 m)
params.h_max      = 3.0;         % Altura maxima geometrica [m]
params.h_min      = 0.15;        % Nivel minimo operacional [m]
params.h_ref      = 2.0;         % Nivel desejado (referencia) [m]

params.ae         = 0.0230749;   % Coeficiente valvula de entrada [m^3/(s*V)]
params.be         = 0.0001068;   % Termo constante da entrada [m^3/s]
params.ks         = 0.0190001;   % Coeficiente valvula de saida [m^3/(s*V*sqrt(m))]

params.u_min      = 2.0;         % Tensao minima das valvulas [V]
params.u_max      = 10.0;        % Tensao maxima das valvulas [V]

params.qe_max_ind = 0.25;        % Vazao maxima individual de entrada [m^3/s] (= 250 L/s)
params.qs_max_ind = 0.3543;      % Vazao maxima individual de saida [m^3/s]   (= 354.3 L/s)

% Calibracao do sensor de nivel: h = alpha_s * Vs + beta_s
params.alpha_s    = 0.30014563;  % [m/V]
params.beta_s     = -0.00142668; % [m]

%% 3. Configuracao do Controlador PQS / SQP e Demanda em LITROS
dt              = 1;    % Periodo de amostragem [s]
Np              = 5;      % Horizonte de predicao (passos)
T_total         = 120;    % Duracao total do ensaio [s]
lambda_u        = 1e-3;   % Regularizacao L2 (Tikhonov) na variacao de comando
M_dem           = 1e6;    % Penalidade para folga de demanda
M_niv           = 1e8;    % Penalidade prioritaria para seguranca de nivel [0.15, 3.00] m

% --- DEFINICAO DA DEMANDA EM LITROS ---
UNIDADE_DEMANDA = 'L/s';  % Opcoes: 'L/s' (litros por segundo) ou 'L/min' (litros por minuto)
USAR_POT_PAINEL = false;  % true: le demanda em L/s do potenciometro Setpoint do Factory I/O (0..10V -> 50..220 L/s)

Q_dem_litros = zeros(T_total, 1);
Q_dem_litros(1:60)   = 100;   % 0s a 59s:  100 L/s (= 0.10 m^3/s)
Q_dem_litros(61:end) = 160;   % 60s a 119s: 160 L/s (= 0.16 m^3/s)

% Conversao automatica de Litros para m^3/s (unidade interna do modelo fisico)
Q_dem_perfil = converter_litros_para_m3s(Q_dem_litros, UNIDADE_DEMANDA);

%% 4. Inicializacao da Conexao Modbus TCP
mb = [];
if USAR_MODBUS
    try
        fprintf('Conectando ao Factory I/O Modbus TCP em %s:%d (Slave ID %d)...\n', ...
                IP_PLANTA, PORTA_MODBUS, SLAVE_ID);
        mb = modbus('tcpip', IP_PLANTA, PORTA_MODBUS, 'Timeout', 2.0);
        cleanupObj = onCleanup(@() encerrar_fio_seguro(mb, HREG_FILL_VALVE, HREG_DISC_VALVE, ...
                                                       COIL_START_LIGHT, COIL_STOP_LIGHT, ESCALA_FIO, SLAVE_ID));

        rodando = read(mb, 'inputs', INP_FIO_RUNNING, 1, SLAVE_ID);
        if ~rodando
            warning('Factory I/O esta pausado/parado (Input 3 = 0). Coloque a simulacao em RUN (F5)!');
        end

        if ATUALIZAR_PAINEL_FIO
            write(mb, 'coils', COIL_START_LIGHT, 1, SLAVE_ID);
            write(mb, 'coils', COIL_STOP_LIGHT,  0, SLAVE_ID);
            write(mb, 'holdingregs', HREG_PANEL_SP, double(round(params.h_ref * ESCALA_FIO)), SLAVE_ID, 'uint16');
        end
        fprintf('Conexao Modbus com Factory I/O estabelecida com sucesso.\n');
    catch ME
        warning('Falha ao conectar em %s:%d (%s). Alternando para simulacao local.', ...
                IP_PLANTA, PORTA_MODBUS, ME.message);
        USAR_MODBUS = false;
    end
end

h_sim = 0.80;
u_ant = [5.0; 5.0];

opts_sqp = optimoptions('fmincon', ...
    'Algorithm', 'sqp', ...
    'Display', 'off', ...
    'MaxIterations', 200, ...
    'MaxFunctionEvaluations', 3000, ...
    'ConstraintTolerance', 1e-7, ...
    'OptimalityTolerance', 1e-7, ...
    'StepTolerance', 1e-8);

hist.t         = (0:T_total-1)' * dt;
hist.h         = zeros(T_total, 1);
hist.vol_L     = zeros(T_total, 1);
hist.Vs        = zeros(T_total, 1);
hist.Vf        = zeros(T_total, 1);
hist.Q_dem_m3s = Q_dem_perfil;
hist.Q_dem_Ls  = Q_dem_perfil * 1000.0;
hist.ue        = zeros(T_total, 1);
hist.us        = zeros(T_total, 1);
hist.qe_Ls     = zeros(T_total, 1);
hist.qs_Ls     = zeros(T_total, 1);
hist.s_dem     = zeros(T_total, 1);
hist.s_niv     = zeros(T_total, 1);
hist.t_cpu     = zeros(T_total, 1);

fprintf('\nIniciando controle otimo SQP (1 Tanque | Ref = %.2f m | Demanda inicial = %.1f %s)...\n', ...
        params.h_ref, Q_dem_litros(1), UNIDADE_DEMANDA);

%% 5. Malha Principal de Controle em Tempo Real
for k = 1:T_total
    t_ciclo = tic;
    Q_dem_atual = Q_dem_perfil(k);

    % 5.1 Leitura dos Input Registers do Factory I/O (Level meter e Flow meter)
    if USAR_MODBUS
        raw_regs = read(mb, 'inputregs', IREG_LEVEL_METER, 3, SLAVE_ID, 'uint16');
        Vs_lido  = double(raw_regs(1)) / ESCALA_FIO;
        Vf_lido  = double(raw_regs(2)) / ESCALA_FIO;
        Vsp_lido = double(raw_regs(3)) / ESCALA_FIO;

        if k == 1 && double(raw_regs(1)) > 1050 && ESCALA_FIO == 100
            warning('Leitura bruta Input Reg 0 = %d (> 1050). Verifique se Scale no Factory I/O esta em 1000!', raw_regs(1));
        end

        % Opcional: le demanda em L/s do potenciometro Setpoint (0..10V mapeado em 50..220 L/s)
        if USAR_POT_PAINEL
            dem_pot_Ls  = 50.0 + (min(max(Vsp_lido, 0.0), 10.0) / 10.0) * 170.0;
            Q_dem_atual = dem_pot_Ls / 1000.0;
        end

        h_atual = max(0.0, params.alpha_s * Vs_lido + params.beta_s);
        if k == 1
            fprintf('Leitura inicial Factory I/O: InputReg0=%d -> Vs=%.3f V -> h0=%.4f m\n', ...
                    raw_regs(1), Vs_lido, h_atual);
        end
    else
        h_atual = h_sim;
        Vs_lido = (h_atual - params.beta_s) / params.alpha_s;
        Vf_lido = 0.0;
    end
    vol_atual = 1000 * params.A * h_atual;

    % 5.2 Formulacao do Problema PQS / SQP no Passo k
    n_vars = 2 * Np + 2;
    lb = [repmat(params.u_min, 2*Np, 1); 0.0; 0.0];
    ub = [repmat(params.u_max, 2*Np, 1); 2.0; 5.0];

    A_ineq = zeros(Np, n_vars);
    b_ineq = (params.qe_max_ind - params.be) * ones(Np, 1);
    for j = 1:Np
        A_ineq(j, j) = params.ae;
    end

    s_niv_init = max(0.0, params.h_min - h_atual + 0.05);
    x0 = [repmat(u_ant(1), Np, 1); repmat(u_ant(2), Np, 1); Q_dem_atual; s_niv_init];

    obj_fun  = @(x) custo_1tanque(x, h_atual, u_ant, Np, dt, params, lambda_u, M_dem, M_niv);
    nonl_fun = @(x) restricoes_1tanque(x, h_atual, Q_dem_atual, Np, dt, params);

    [x_opt, ~, exitflag] = fmincon(obj_fun, x0, A_ineq, b_ineq, [], [], lb, ub, nonl_fun, opts_sqp);

    if exitflag <= 0
        warning('Passo %d: fmincon exitflag = %d.', k, exitflag);
    end

    ue_cmd    = double(min(max(x_opt(1),    params.u_min), params.u_max));
    us_cmd    = double(min(max(x_opt(Np+1), params.u_min), params.u_max));
    s_dem_opt = x_opt(end-1);
    s_niv_opt = x_opt(end);

    % 5.3 Escrita nos Holding Registers do Factory I/O (Fill valve e Discharge valve)
    if USAR_MODBUS
        vals_valvulas = double([round(ue_cmd * ESCALA_FIO), round(us_cmd * ESCALA_FIO)]);
        write(mb, 'holdingregs', HREG_FILL_VALVE, vals_valvulas, SLAVE_ID, 'uint16');

        if ATUALIZAR_PAINEL_FIO
            try
                write(mb, 'holdingregs', HREG_PANEL_PV, double(round(h_atual * ESCALA_FIO)), SLAVE_ID, 'uint16');
            catch
            end
        end
    else
        h_sim = passo_rk4_1tanque(h_sim, ue_cmd, us_cmd, dt, params);
    end

    qe_atual_Ls = (params.ae * ue_cmd + params.be) * 1000.0;
    qs_atual_Ls = (params.ks * us_cmd * sqrt(max(h_atual, 1e-6))) * 1000.0;

    hist.h(k)         = h_atual;
    hist.vol_L(k)     = vol_atual;
    hist.Vs(k)        = Vs_lido;
    hist.Vf(k)        = Vf_lido;
    hist.Q_dem_m3s(k) = Q_dem_atual;
    hist.Q_dem_Ls(k)  = Q_dem_atual * 1000.0;
    hist.ue(k)        = ue_cmd;
    hist.us(k)        = us_cmd;
    hist.qe_Ls(k)     = qe_atual_Ls;
    hist.qs_Ls(k)     = qs_atual_Ls;
    hist.s_dem(k)     = s_dem_opt;
    hist.s_niv(k)     = s_niv_opt;
    hist.t_cpu(k)     = toc(t_ciclo);

    u_ant = [ue_cmd; us_cmd];

    if mod(k-1, 10) == 0 || k == T_total
        fprintf('t=%3.0fs | h=%.4fm (Vol=%6.1fL) | Qdem=%5.1f L/s | qs=%5.1f L/s | ue=%.3fV | us=%.3fV | cpu=%.1fms\n', ...
                hist.t(k), h_atual, vol_atual, hist.Q_dem_Ls(k), qs_atual_Ls, ue_cmd, us_cmd, hist.t_cpu(k)*1000);
    end

    if USAR_MODBUS
        pausa = dt - toc(t_ciclo);
        if pausa > 0
            pause(pausa);
        end
    end
end

%% 6. Metricas de Desempenho e Visualizacao
rmse_nivel  = sqrt(mean((hist.h - params.h_ref).^2));
erro_dem_Ls = max(abs(hist.qs_Ls - hist.Q_dem_Ls));
fprintf('\n=== RESUMO DO CONTROLE (1 TANQUE) ===\n');
fprintf('RMSE Nivel vs Referencia (2.0 m): %.5f m\n', rmse_nivel);
fprintf('Erro Maximo de Atendimento a Demanda: %.3f L/s\n', erro_dem_Ls);
fprintf('Nivel Min/Max atingido: [%.4f m, %.4f m] (Limites: [%.2f, %.2f] m)\n', ...
        min(hist.h), max(hist.h), params.h_min, params.h_max);

figure('Name', 'Controle Otimo SQP - 1 Tanque (Factory I/O)', 'Color', 'w');
subplot(3,1,1);
plot(hist.t, hist.h, 'b-', 'LineWidth', 1.8); hold on;
yline(params.h_ref, 'r--', 'Referencia (2.0 m)', 'LineWidth', 1.2);
yline(params.h_min, 'k:',  'Nivel Min (0.15 m)');
ylabel('Nivel h [m]'); grid on; title('Controle PQS/SQP - 1 Tanque Factory I/O (192.168.15.117:502)');

subplot(3,1,2);
plot(hist.t, hist.Q_dem_Ls, 'k--', 'LineWidth', 1.5); hold on;
plot(hist.t, hist.qs_Ls, 'g-', 'LineWidth', 1.5);
plot(hist.t, hist.qe_Ls, 'm-.', 'LineWidth', 1.3);
ylabel('Vazao [L/s]'); legend('Demanda Q_{dem} [L/s]', 'Saida q_s [L/s]', 'Entrada q_e [L/s]', 'Location', 'best'); grid on;

subplot(3,1,3);
plot(hist.t, hist.ue, 'b-', 'LineWidth', 1.5); hold on;
plot(hist.t, hist.us, 'r-', 'LineWidth', 1.5);
yline(params.u_min, 'k:'); yline(params.u_max, 'k:');
ylabel('Tensao [V]'); xlabel('Tempo [s]'); legend('u_e (Fill valve)', 'u_s (Discharge valve)', 'Location', 'best'); grid on;

%% ==================== FUNCOES AUXILIARES ====================

function q_m3s = converter_litros_para_m3s(q_litros, unidade)
    if strcmpi(unidade, 'L/s')
        q_m3s = q_litros / 1000.0;
    elseif strcmpi(unidade, 'L/min')
        q_m3s = q_litros / 60000.0;
    else
        error('Unidade invalida: "%s". Use ''L/s'' ou ''L/min''.', unidade);
    end
end

function [h_pred, qe_pred, qs_pred] = simular_horizonte_1tanque(x, h0, Np, dt, p)
    ue = x(1:Np);
    us = x(Np+1:2*Np);
    h_pred  = zeros(Np+1, 1);
    qe_pred = p.ae * ue + p.be;
    qs_pred = zeros(Np, 1);
    h_pred(1) = h0;
    for j = 1:Np
        h_seguro = max(h_pred(j), 1e-6);
        qs_pred(j) = p.ks * us(j) * sqrt(h_seguro);
        h_pred(j+1) = h_pred(j) + (dt / p.A) * (qe_pred(j) - qs_pred(j));
    end
end

function J = custo_1tanque(x, h0, u_ant, Np, dt, p, lambda_u, M_dem, M_niv)
    ue    = x(1:Np);
    us    = x(Np+1:2*Np);
    s_dem = x(end-1);
    s_niv = x(end);
    [h_pred, ~, ~] = simular_horizonte_1tanque(x, h0, Np, dt, p);

    erro_quad_h = sum((h_pred(2:end) - p.h_ref).^2);
    due = diff([u_ant(1); ue]);
    dus = diff([u_ant(2); us]);
    reg = lambda_u * sum(due.^2 + dus.^2);

    J = erro_quad_h + reg + M_dem * (s_dem^2) + M_niv * (s_niv^2);
end

function [c, ceq] = restricoes_1tanque(x, h0, Q_dem, Np, dt, p)
    s_dem = x(end-1);
    s_niv = x(end);
    [h_pred, ~, qs_pred] = simular_horizonte_1tanque(x, h0, Np, dt, p);

    c_hmin   = p.h_min - (h_pred(2:end) + s_niv);
    c_hmax   = (h_pred(2:end) - s_niv) - p.h_max;
    c_qsmax  = qs_pred - p.qs_max_ind;
    c_dem_up = qs_pred - (Q_dem + s_dem);
    c_dem_lo = (Q_dem - s_dem) - qs_pred;

    c   = [c_hmin; c_hmax; c_qsmax; c_dem_up; c_dem_lo];
    ceq = [];
end

function h_next = passo_rk4_1tanque(h, ue, us, dt, p)
    f = @(hv) ((p.ae * ue + p.be) - p.ks * us * sqrt(max(hv, 1e-6))) / p.A;
    k1 = f(h);
    k2 = f(h + 0.5*dt*k1);
    k3 = f(h + 0.5*dt*k2);
    k4 = f(h + dt*k3);
    h_next = max(0.0, h + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4));
end

function encerrar_fio_seguro(mb, hreg_fill, hreg_disc, coil_start, coil_stop, escala, slave_id)
    if ~isempty(mb)
        try
            val_min = double(round(2.0 * escala));
            write(mb, 'holdingregs', hreg_fill, [val_min, val_min], slave_id, 'uint16');
            write(mb, 'coils', coil_start, 0, slave_id);
            write(mb, 'coils', coil_stop,  1, slave_id);
            clear mb;
            fprintf('Conexao Factory I/O encerrada com seguranca (valvulas em 2.0 V).\n');
        catch
        end
    end
end

---
# 2. Controle de dois tanques

Arquivo de origem: `controle_2_tanques_v2(1).m`.

O script configura dois tanques com alimentação compartilhada, prioridades entre os reservatórios, perfil de demanda por fases e otimização SQP com regularização dos comandos. Em cada ciclo, lê os níveis, atualiza a demanda programada, resolve o problema e aplica os comandos às válvulas.

In [ ]:
%% controle_2_tanques_v2.m
% Projeto COS360 - Otimizacao (PESC/COPPE/UFRJ)
% Controle Otimo Nao Linear (PQS / SQP) para 2 Tanques com Entrada Compartilhada (250 L/s)
% e Abdicacao Automatica da Demanda do Tanque 2 para Priorizar h1 = 2.0 m
% Servidor Modbus TCP/IP Factory I/O: 192.168.15.117:502 | Slave ID: 1
%
% Hierarquia de Prioridades Implementada no SQP:
%   1) Seguranca fisica dos niveis: h1, h2 in [0.15, 3.00] m         (M_niv  = 1e8)
%   2) Atendimento obrigatorio a demanda do Tanque 1: qs1 = Q_dem1   (M_dem1 = 1e6)
%   3) Manutencao da referencia h1 = 2.0 m no Tanque 1 (prioritario) (w_h1   = 100.0)
%   4) Manutencao da referencia h2 = 2.0 m no Tanque 2 (secundario)  (w_h2   = 10.0)
%   5) Demanda opcional/abdicavel do Tanque 2: qs2 <= Q_dem2         (w_dem2 = 20.0)
%      -> Se a entrada constante (250 L/s) nao bastar para suprir ambas as demandas
%         e manter h1 = 2.0 m e h2 = 2.0 m, o otimizador reduz qs2 (abdicando parcial
%         ou totalmente da demanda do Tanque 2) para liberar vazao da valvula mestre
%         de entrada ao Tanque 1.
%
% Mapa Modbus TCP Factory I/O (enderecos MATLAB = base 1):
%   Tanque 1 (Central - Prioritario):
%     - Nivel:  Input Reg 0  -> end. MATLAB 1 ('inputregs')   : Level meter
%     - Entr.:  Holding Reg 0-> end. MATLAB 1 ('holdingregs') : Fill valve
%     - Saida:  Holding Reg 1-> end. MATLAB 2 ('holdingregs') : Discharge valve
%   Tanque 2 (Esquerda - Tank 0, Secundario):
%     - Nivel:  Input Reg 5  -> end. MATLAB 6 ('inputregs')   : Tank 0 (Level Meter)
%     - Entr.:  Holding Reg 5-> end. MATLAB 6 ('holdingregs') : Tank 0 (Fill Valve)
%     - Saida:  Holding Reg 6-> end. MATLAB 7 ('holdingregs') : Tank 0 (Discharge Valve)

clear; clc; close all;

%% 1. Configuracao de Comunicacao Modbus TCP (Factory I/O)
IP_PLANTA            = '192.168.15.117';
PORTA_MODBUS         = 502;
SLAVE_ID             = 1;
USAR_MODBUS          = true;   % true: conecta ao Factory I/O | false: simulacao local RK4

ESCALA_FIO           = 100.0;
ATUALIZAR_PAINEL_FIO = true;

% Enderecos Modbus MATLAB (base 1): [Tanque 1 (Central), Tanque 2 (Tank 0)]
IREGS_LEVEL_METERS   = [1, 6]; % Input Reg 0 (Central) e Input Reg 5 (Tank 0)
HREGS_FILL_VALVES    = [1, 6]; % Holding Reg 0 (Central) e Holding Reg 5 (Tank 0)
HREGS_DISC_VALVES    = [2, 7]; % Holding Reg 1 (Central) e Holding Reg 6 (Tank 0)

INP_FIO_RUNNING      = 4;      % Input 3: FACTORY I/O (Running)
HREG_PANEL_SP        = 3;      % Holding Reg 2: SP
HREG_PANEL_PV        = 4;      % Holding Reg 3: PV
COIL_START_LIGHT     = 1;      % Coil 0: Start light
COIL_STOP_LIGHT      = 3;      % Coil 2: Stop light

%% 2. Parametros Fisicos e Restricoes da Planta (Modelo Matematico)
N_TANQUES         = 2;
params.N          = N_TANQUES;
params.A          = pi;          % Area transversal de cada tanque [m^2] (diametro = 2.0 m)
params.h_max      = 3.0;         % Altura maxima geometrica [m]
params.h_min      = 0.15;        % Nivel minimo operacional [m]
params.h_ref      = 2.0;         % Nivel de referencia [m]

params.ae         = 0.0230749;   % Coeficiente valvula de entrada [m^3/(s*V)]
params.be         = 0.0001068;   % Termo constante da entrada [m^3/s]
params.ks         = 0.0190001;   % Coeficiente valvula de saida [m^3/(s*V*sqrt(m))]

params.u_min      = 2.0;         % Tensao minima das valvulas [V]
params.u_max      = 10.0;        % Tensao maxima das valvulas [V]

params.qe_max_ind = 0.25;        % Vazao maxima individual de entrada [m^3/s] (= 250 L/s)
params.qe_max_tot = 0.40;        % Valvula de entrada constante / coletor comum [m^3/s] (= 250 L/s)
params.qs_max_ind = 0.3543;      % Vazao maxima individual de saida [m^3/s] (= 354.3 L/s)

% Calibracao do sensor de nivel: h_i = alpha_s * Vs_i + beta_s
params.alpha_s    = 0.30014563;  % [m/V]
params.beta_s     = -0.00142668; % [m]

%% 3. Configuracao do Controlador PQS / SQP, Hierarquia de Pesos e Demandas em LITROS
dt              = 1.0;    % Periodo de amostragem [s]
Np              = 5;      % Horizonte de predicao [passos]
T_total         = 180;    % Duracao total do ensaio [s]

% Pesos da funcao objetivo (hierarquia de prioridades):
w_h1            = 100.0;  % Prioridade alta: manter Tanque 1 em h_ref = 2.0 m
w_h2            = 10.0;   % Prioridade media: manter Tanque 2 em h_ref = 2.0 m (menor que T1)
w_dem2          = 20.0;   % Prioridade baixa: deficit de demanda do Tanque 2 (abdicavel)
params.w_h1     = w_h1;
params.w_h2     = w_h2;
params.w_dem2   = w_dem2;

lambda_u        = 1e-2;   % Regularizacao L2 (Tikhonov) na variacao de comando
M_dem1          = 1e6;    % Penalidade alta: demanda do Tanque 1 e obrigatoria
M_niv           = 1e8;    % Penalidade maxima: limites de seguranca [0.15, 3.00] m

% --- DEFINICAO DAS DEMANDAS INDEPENDENTES EM LITROS ---
UNIDADE_DEMANDA = 'L/s';  % Opcoes: 'L/s' (litros por segundo) ou 'L/min' (litros por minuto)

Q_dem1_litros = zeros(T_total, 1); % Demanda obrigatoria do Tanque 1
Q_dem2_litros = zeros(T_total, 1); % Demanda opcional/abdicavel do Tanque 2

% Fase 1 (0s a 59s): Soma = 90 + 80 = 170 L/s (<= 250 L/s -> ambos atingem 2.0 m e cumprem demandas)
Q_dem1_litros(1:60)    = 90;
Q_dem2_litros(1:60)    = 80;

% Fase 2 (60s a 119s): Soma = 240 + 180 = 440 L/s (> 250 L/s -> entrada satura em 250 L/s;
%   Tanque 2 abdica parcialmente de sua demanda para que Tanque 1 mantenha 2.0 m com folga)
Q_dem1_litros(61:120)  = 230;
Q_dem2_litros(61:120)  = 180;

% Fase 3 (120s a 179s): Soma = 80 + 70 = 150 L/s (<= 250 L/s -> Tanque 2 retoma 100% da demanda)
Q_dem1_litros(121:end) = 90;
Q_dem2_litros(121:end) = 80;

% Conversao automatica de Litros para m^3/s
Q_dem1_perfil = converter_litros_para_m3s(Q_dem1_litros, UNIDADE_DEMANDA);
Q_dem2_perfil = converter_litros_para_m3s(Q_dem2_litros, UNIDADE_DEMANDA);

%% 4. Inicializacao da Conexao Modbus TCP
mb = [];
if USAR_MODBUS
    try
        fprintf('Conectando ao Factory I/O (2 Tanques) em %s:%d (Slave ID %d)...\n', ...
                IP_PLANTA, PORTA_MODBUS, SLAVE_ID);
        mb = modbus('tcpip', IP_PLANTA, PORTA_MODBUS, 'Timeout', 2.0);
        cleanupObj = onCleanup(@() encerrar_fio_2t_seguro(mb, HREGS_FILL_VALVES, HREGS_DISC_VALVES, ...
                                                          COIL_START_LIGHT, COIL_STOP_LIGHT, ESCALA_FIO, SLAVE_ID));

        rodando = read(mb, 'inputs', INP_FIO_RUNNING, 1, SLAVE_ID);
        if ~rodando
            warning('Factory I/O esta pausado/parado (Input 3 = 0). Coloque a simulacao em RUN (F5)!');
        end

        if ATUALIZAR_PAINEL_FIO
            write(mb, 'coils', COIL_START_LIGHT, 1, SLAVE_ID);
            write(mb, 'coils', COIL_STOP_LIGHT,  0, SLAVE_ID);
            write(mb, 'holdingregs', HREG_PANEL_SP, double(round(params.h_ref * ESCALA_FIO)), SLAVE_ID, 'uint16');
        end
        fprintf('Conexao Modbus (2 Tanques) estabelecida com sucesso.\n');
    catch ME
        warning('Falha ao conectar em %s:%d (%s). Alternando para simulacao local.', ...
                IP_PLANTA, PORTA_MODBUS, ME.message);
        USAR_MODBUS = false;
    end
end

h_sim = [0.80; 0.80];
u_ant = [5.0; 5.0; 4.0; 4.0]; % [ue1; ue2; us1; us2]

opts_sqp = optimoptions('fmincon', ...
    'Algorithm', 'sqp', ...
    'Display', 'off', ...
    'MaxIterations', 250, ...
    'MaxFunctionEvaluations', 5000, ...
    'ConstraintTolerance', 1e-7, ...
    'OptimalityTolerance', 1e-7, ...
    'StepTolerance', 1e-8);

hist.t            = (0:T_total-1)' * dt;
hist.h            = zeros(T_total, N_TANQUES);
hist.vol_L        = zeros(T_total, N_TANQUES);
hist.Q_dem1_Ls    = Q_dem1_perfil * 1000.0;
hist.Q_dem2_Ls    = Q_dem2_perfil * 1000.0;
hist.ue           = zeros(T_total, N_TANQUES);
hist.us           = zeros(T_total, N_TANQUES);
hist.qe_Ls        = zeros(T_total, N_TANQUES);
hist.qs_Ls        = zeros(T_total, N_TANQUES);
hist.dem2_abd_Ls  = zeros(T_total, 1); % Parcela da demanda do Tanque 2 abdicada [L/s]
hist.s_dem1       = zeros(T_total, 1);
hist.s_niv        = zeros(T_total, 1);
hist.t_cpu        = zeros(T_total, 1);

% Variaveis de decisao do SQP:
%   x(1 : 4*Np)            -> comandos [ue1_j, ue2_j, us1_j, us2_j] no horizonte j=1..Np
%   x(4*Np + 1)            -> s_dem1 (folga de demanda obrigatoria do Tanque 1)
%   x(4*Np + 2 : 5*Np + 1) -> s_dem2(1..Np) (deficit opcional de demanda do Tanque 2)
%   x(5*Np + 2)            -> s_niv (folga de seguranca de nivel [0.15, 3.00] m)
n_u    = 2 * N_TANQUES; % 4 controles por passo
n_vars = n_u * Np + 1 + Np + 1;

lb = [repmat(params.u_min, n_u * Np, 1); 0.0; zeros(Np, 1); 0.0];
ub = [repmat(params.u_max, n_u * Np, 1); 2.0; 2.0 * ones(Np, 1); 5.0];

% Restricoes lineares no horizonte:
%   1) Vazao individual de entrada: ae * ue_i <= qe_max_ind - be  (2*Np linhas)
%   2) Valvula constante de entrada: ae * (ue1 + ue2) <= 250 L/s - 2*be (Np linhas)
n_lin  = (N_TANQUES + 1) * Np;
A_ineq = zeros(n_lin, n_vars);
b_ineq = zeros(n_lin, 1);
row_idx = 0;
for j = 1:Np
    base = (j - 1) * n_u;
    for i = 1:N_TANQUES
        row_idx = row_idx + 1;
        A_ineq(row_idx, base + i) = params.ae;
        b_ineq(row_idx) = params.qe_max_ind - params.be;
    end
    row_idx = row_idx + 1;
    A_ineq(row_idx, base + (1:N_TANQUES)) = params.ae;
    b_ineq(row_idx) = params.qe_max_tot - N_TANQUES * params.be;
end

fprintf('\nIniciando SQP (2 Tanques | w_h1=%.1f, w_h2=%.1f, w_dem2=%.1f | Q_e_max=250 L/s)...\n', ...
        w_h1, w_h2, w_dem2);

%% 5. Malha Principal de Controle em Tempo Real
for k = 1:T_total
    t_ciclo = tic;
    Q_dem_atual = [Q_dem1_perfil(k); Q_dem2_perfil(k)];

    % 5.1 Leitura dos niveis dos 2 tanques no Factory I/O
    h_atual = zeros(N_TANQUES, 1);
    if USAR_MODBUS
        bloco_iregs = read(mb, 'inputregs', 1, 6, SLAVE_ID, 'uint16');
        for i = 1:N_TANQUES
            raw_i = double(bloco_iregs(IREGS_LEVEL_METERS(i)));
            Vs_i  = raw_i / ESCALA_FIO;
            h_atual(i) = max(0.0, params.alpha_s * Vs_i + params.beta_s);
        end
        if k == 1
            fprintf('Niveis iniciais Factory I/O [Tanque 1, Tanque 2]: [%.4f, %.4f] m\n', ...
                    h_atual(1), h_atual(2));
        end
    else
        h_atual = h_sim;
    end
    vol_atual = 1000 * params.A * h_atual;

    % 5.2 Resolucao do Problema PQS / SQP
    s_niv_init = max(0.0, params.h_min - min(h_atual) + 0.05);
    x0 = [repmat(u_ant, Np, 1); 0.0; zeros(Np, 1); s_niv_init];

    obj_fun  = @(x) custo_2tanques(x, h_atual, u_ant, Np, dt, params, lambda_u, M_dem1, M_niv);
    nonl_fun = @(x) restricoes_2tanques(x, h_atual, Q_dem_atual, Np, dt, params);

    [x_opt, ~, exitflag] = fmincon(obj_fun, x0, A_ineq, b_ineq, [], [], lb, ub, nonl_fun, opts_sqp);

    if exitflag <= 0
        warning('Passo %d: fmincon exitflag = %d.', k, exitflag);
    end

    u_cmd      = double(min(max(x_opt(1:n_u), params.u_min), params.u_max));
    ue_cmd     = u_cmd(1:2); % [ue1; ue2]
    us_cmd     = u_cmd(3:4); % [us1; us2]
    s_dem1_opt = x_opt(n_u*Np + 1);
    s_niv_opt  = x_opt(end);

    % 5.3 Escrita dos 4 Comandos nas Valvulas do Factory I/O
    if USAR_MODBUS
        write(mb, 'holdingregs', HREGS_FILL_VALVES(1), double(round([ue_cmd(1), us_cmd(1)] * ESCALA_FIO)), SLAVE_ID, 'uint16');
        write(mb, 'holdingregs', HREGS_FILL_VALVES(2), double(round([ue_cmd(2), us_cmd(2)] * ESCALA_FIO)), SLAVE_ID, 'uint16');

        if ATUALIZAR_PAINEL_FIO
            try
                write(mb, 'holdingregs', HREG_PANEL_PV, double(round(h_atual(1) * ESCALA_FIO)), SLAVE_ID, 'uint16');
            catch
            end
        end
    else
        for i = 1:N_TANQUES
            h_sim(i) = passo_rk4_tanque(h_sim(i), ue_cmd(i), us_cmd(i), dt, params);
        end
    end

    qe_atual_Ls = (params.ae * ue_cmd + params.be) * 1000.0;
    qs_atual_Ls = (params.ks * us_cmd .* sqrt(max(h_atual, 1e-6))) * 1000.0;
    abd2_Ls     = max(0.0, hist.Q_dem2_Ls(k) - qs_atual_Ls(2));

    hist.h(k, :)       = h_atual';
    hist.vol_L(k, :)   = vol_atual';
    hist.ue(k, :)      = ue_cmd';
    hist.us(k, :)      = us_cmd';
    hist.qe_Ls(k, :)   = qe_atual_Ls';
    hist.qs_Ls(k, :)   = qs_atual_Ls';
    hist.dem2_abd_Ls(k)= abd2_Ls;
    hist.s_dem1(k)     = s_dem1_opt;
    hist.s_niv(k)      = s_niv_opt;
    hist.t_cpu(k)      = toc(t_ciclo);

    u_ant = u_cmd;

    if mod(k-1, 10) == 0 || k == T_total
        fprintf('t=%3.0fs | h=[%.3f, %.3f]m | qe=[%5.1f,%5.1f] (sum=%5.1f) | qs1=%5.1f/%5.1f | qs2=%5.1f/%5.1f (abd=%4.1f L/s)\n', ...
                hist.t(k), h_atual(1), h_atual(2), ...
                qe_atual_Ls(1), qe_atual_Ls(2), sum(qe_atual_Ls), ...
                qs_atual_Ls(1), hist.Q_dem1_Ls(k), ...
                qs_atual_Ls(2), hist.Q_dem2_Ls(k), abd2_Ls);
    end

    if USAR_MODBUS
        pausa = dt - toc(t_ciclo);
        if pausa > 0
            pause(pausa);
        end
    end
end

%% 6. Metricas de Desempenho e Graficos
rmse_t1           = sqrt(mean((hist.h(:,1) - params.h_ref).^2));
rmse_t2           = sqrt(mean((hist.h(:,2) - params.h_ref).^2));
max_qe_coletor_Ls = max(sum(hist.qe_Ls, 2));
erro_dem1_Ls      = max(abs(hist.qs_Ls(:,1) - hist.Q_dem1_Ls));
max_abd2_Ls       = max(hist.dem2_abd_Ls);

fprintf('\n=== RESUMO DO CONTROLE (2 TANQUES COM ABDICACAO DE DEMANDA NO T2) ===\n');
fprintf('RMSE Nivel Tanque 1 (Prioritario, w_h1=%.1f) vs 2.0 m: %.5f m (Final: %.4f m)\n', w_h1, rmse_t1, hist.h(end,1));
fprintf('RMSE Nivel Tanque 2 (Secundario,  w_h2=%.1f) vs 2.0 m: %.5f m (Final: %.4f m)\n', w_h2, rmse_t2, hist.h(end,2));
fprintf('Pico Vazao Total Entrada (Limite <= 250.0 L/s): %.2f L/s\n', max_qe_coletor_Ls);
fprintf('Erro Maximo Demanda Obrigatoria Tanque 1: %.3f L/s\n', erro_dem1_Ls);
fprintf('Maxima Abdicacao de Demanda no Tanque 2 (para proteger h1=2m): %.2f L/s\n', max_abd2_Ls);
fprintf('Nivel Minimo Observado (Seguranca >= 0.15 m): %.4f m\n', min(hist.h(:)));

figure('Name', 'Controle Otimo SQP - 2 Tanques (Prioridade h1=2m e Abdicacao Demanda T2)', 'Color', 'w');
subplot(3,1,1);
plot(hist.t, hist.h(:,1), 'b-', 'LineWidth', 1.8); hold on;
plot(hist.t, hist.h(:,2), 'm-', 'LineWidth', 1.8);
yline(params.h_ref, 'r--', 'Referencia (2.0 m)', 'LineWidth', 1.2);
yline(params.h_min, 'k:',  'Nivel Minimo (0.15 m)', 'LineWidth', 1.2);
ylabel('Nivel h_i [m]');
legend(sprintf('Tanque 1 (Prioritario, w_{h1}=%.0f)', w_h1), ...
       sprintf('Tanque 2 (Secundario, w_{h2}=%.0f)', w_h2), 'Location', 'best');
grid on; title('Controle SQP - 2 Tanques com Entrada Constante (250 L/s) e Abdicacao de Demanda no T2');

subplot(3,1,2);
plot(hist.t, hist.Q_dem1_Ls, 'b--', 'LineWidth', 1.4); hold on;
plot(hist.t, hist.qs_Ls(:,1), 'b-',  'LineWidth', 1.6);
plot(hist.t, hist.Q_dem2_Ls, 'm--', 'LineWidth', 1.4);
plot(hist.t, hist.qs_Ls(:,2), 'm-',  'LineWidth', 1.6);
plot(hist.t, sum(hist.qe_Ls, 2), 'k-', 'LineWidth', 1.8);
yline(params.qe_max_tot * 1000.0, 'r:', 'Valvula Entrada (250 L/s)', 'LineWidth', 1.3);
ylabel('Vazao [L/s]');
legend('Demanda T1 (Obrig.)', 'Saida q_{s1}', 'Demanda T2 (Opc.)', 'Saida q_{s2}', 'Entrada Total q_{e1}+q_{e2}', 'Location', 'eastoutside');
grid on;

subplot(3,1,3);
plot(hist.t, hist.ue(:,1), 'b-',  'LineWidth', 1.5); hold on;
plot(hist.t, hist.ue(:,2), 'm-',  'LineWidth', 1.5);
plot(hist.t, hist.us(:,1), 'b--', 'LineWidth', 1.4);
plot(hist.t, hist.us(:,2), 'm--', 'LineWidth', 1.4);
ylabel('Comandos [V]'); xlabel('Tempo [s]');
legend('u_{e1} (Entr. T1)', 'u_{e2} (Entr. T2)', 'u_{s1} (Saida T1)', 'u_{s2} (Saida T2)', 'Location', 'eastoutside');
grid on;

%% ==================== FUNCOES AUXILIARES ====================

function q_m3s = converter_litros_para_m3s(q_litros, unidade)
    if strcmpi(unidade, 'L/s')
        q_m3s = q_litros / 1000.0;
    elseif strcmpi(unidade, 'L/min')
        q_m3s = q_litros / 60000.0;
    else
        error('Unidade invalida: "%s". Use ''L/s'' ou ''L/min''.', unidade);
    end
end

function [h_pred, qe_pred, qs_pred] = simular_horizonte_2tanques(x, h0, Np, dt, p)
    N = p.N;
    n_u = 2 * N;
    h_pred  = zeros(Np+1, N);
    qe_pred = zeros(Np, N);
    qs_pred = zeros(Np, N);
    h_pred(1, :) = h0';

    for j = 1:Np
        base = (j - 1) * n_u;
        ue = x(base + (1:N))';
        us = x(base + N + (1:N))';
        qe_pred(j, :) = p.ae * ue + p.be;
        h_seguro = max(h_pred(j, :), 1e-6);
        qs_pred(j, :) = p.ks * us .* sqrt(h_seguro);
        h_pred(j+1, :) = h_pred(j, :) + (dt / p.A) * (qe_pred(j, :) - qs_pred(j, :));
    end
end

function J = custo_2tanques(x, h0, u_ant, Np, dt, p, lambda_u, M_dem1, M_niv)
    N      = p.N;
    n_u    = 2 * N;
    s_dem1 = x(n_u*Np + 1);
    s_dem2 = x(n_u*Np + 2 : n_u*Np + 1 + Np);
    s_niv  = x(end);
    [h_pred, ~, ~] = simular_horizonte_2tanques(x, h0, Np, dt, p);

    erro_quad_h1 = p.w_h1 * sum((h_pred(2:end, 1) - p.h_ref).^2);
    erro_quad_h2 = p.w_h2 * sum((h_pred(2:end, 2) - p.h_ref).^2);
    custo_dem2   = p.w_dem2 * sum(s_dem2.^2);

    reg = 0.0;
    u_prev = u_ant;
    for j = 1:Np
        base = (j - 1) * n_u;
        u_curr = x(base + (1:n_u));
        reg = reg + lambda_u * sum((u_curr - u_prev).^2);
        u_prev = u_curr;
    end

    J = erro_quad_h1 + erro_quad_h2 + custo_dem2 + reg + M_dem1 * (s_dem1^2) + M_niv * (s_niv^2);
end

function [c, ceq] = restricoes_2tanques(x, h0, Q_dem, Np, dt, p)
    n_u    = 2 * p.N;
    s_dem1 = x(n_u*Np + 1);
    s_dem2 = x(n_u*Np + 2 : n_u*Np + 1 + Np);
    s_niv  = x(end);
    [h_pred, ~, qs_pred] = simular_horizonte_2tanques(x, h0, Np, dt, p);

    h_fut = h_pred(2:end, :);

    c_hmin    = p.h_min - (h_fut(:) + s_niv);
    c_hmax    = (h_fut(:) - s_niv) - p.h_max;
    c_qs_ind  = qs_pred(:) - p.qs_max_ind;

    % Tanque 1: demanda obrigatoria (qs1 == Q_dem1)
    c_dem1_up = qs_pred(:, 1) - (Q_dem(1) + s_dem1);
    c_dem1_lo = (Q_dem(1) - s_dem1) - qs_pred(:, 1);

    % Tanque 2: demanda opcional/abdicavel (qs2 <= Q_dem2 e deficit s_dem2 >= Q_dem2 - qs2)
    c_dem2_up = qs_pred(:, 2) - Q_dem(2);
    c_dem2_lo = (Q_dem(2) - s_dem2) - qs_pred(:, 2);

    c   = [c_hmin; c_hmax; c_qs_ind; c_dem1_up; c_dem1_lo; c_dem2_up; c_dem2_lo];
    ceq = [];
end

function h_next = passo_rk4_tanque(h, ue, us, dt, p)
    f = @(hv) ((p.ae * ue + p.be) - p.ks * us * sqrt(max(hv, 1e-6))) / p.A;
    k1 = f(h);
    k2 = f(h + 0.5*dt*k1);
    k3 = f(h + 0.5*dt*k2);
    k4 = f(h + dt*k3);
    h_next = max(0.0, h + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4));
end

function encerrar_fio_2t_seguro(mb, hregs_fill, hregs_disc, coil_start, coil_stop, escala, slave_id)
    if ~isempty(mb)
        try
            val_min = double(round(2.0 * escala));
            for i = 1:length(hregs_fill)
                write(mb, 'holdingregs', hregs_fill(i), val_min, slave_id, 'uint16');
                write(mb, 'holdingregs', hregs_disc(i), val_min, slave_id, 'uint16');
            end
            write(mb, 'coils', coil_start, 0, slave_id);
            write(mb, 'coils', coil_stop,  1, slave_id);
            clear mb;
            fprintf('Conexao Factory I/O (2 Tanques) encerrada com seguranca (valvulas em 2.0 V).\n');
        catch
        end
    end
end


---
## Observações finais
- Os dois códigos usam `fmincon` com algoritmo `sqp`; são experimentos separados.
- O horizonte de previsão, o período de amostragem, os pesos e as demandas estão definidos em cada script.
- A simulação local (`USAR_MODBUS = false`) usa o modelo matemático implementado no código; a execução conectada (`USAR_MODBUS = true`) lê os sensores e escreve nos registradores do Factory I/O.